# Velocity and growth

Calculate velocity components and growth rates from a trained model, and save
the arrays and per-cell values for downstream analysis.
The example uses the small chicken-heart dataset. With your own trained model,
start at **Calculate velocity components** using the variables at the end of
[Train a model](../training.md). No preprocessing or training is repeated here.

## Load the example data and model

Launch JupyterLab as described in [Installation](../installation.md). The first
cell uses the extracted chicken-heart files, downloading them only if needed.

In [1]:
import os
from pathlib import Path
import anndata as ad
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import CytoBridge as cb

PROJECT_DIR = Path(os.environ.get("CYTOBRIDGE_PROJECT_DIR", ".")).resolve()
DATA_DIR = PROJECT_DIR / "data/chicken_heart"
OUTPUT_DIR = PROJECT_DIR / "outputs/model_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if not (DATA_DIR / "aligned.h5ad").is_file() or not (DATA_DIR / "model/config.yaml").is_file():
    cb.datasets.download("chicken_heart", destination=PROJECT_DIR)
adata = ad.read_h5ad(DATA_DIR / "aligned.h5ad", backed="r")
states = cb.tl.model_state_adata(adata)
adata.file.close()
time_key = "time_point_processed"
annotation_key = "celltype_prediction"
observed_times = sorted(states.obs[time_key].unique())
model = cb.tl.load_dynamical_model_from_dir(
    DATA_DIR / "model", dim=states.n_vars, device=DEVICE,
    edge_predictor_path=DATA_DIR / "edge_classifier/chicken_heart_edge_model.pt",
).model
pd.Series({"cells": states.n_obs, "state dimensions": states.n_vars, "device": DEVICE})

cells               3550
state dimensions      52
device              cuda
dtype: object

## Calculate velocity components

Choose one observed stage. The model states contain two spatial coordinates
followed by the expression PCA features, in the same order used for training.
`compute_velocity_components` returns one array per component, with one row
per cell. The full field combines intrinsic drift, interaction and the score
term. Set the random seed before evaluating the interaction groups.

In [2]:
time = observed_times[1]
population = states[np.isclose(states.obs[time_key], time)].copy()
cb.tl.set_global_random_seed(42)
velocity = cb.tl.compute_velocity_components(
    population.X, float(time), model, device=DEVICE,
    interaction_m=1024, interaction_threshold=model.interaction_net.cutoff,
)
np.savez_compressed(OUTPUT_DIR / "velocity_components.npz",
                    cell_ids=population.obs_names.to_numpy(dtype=str),
                    time=float(time), **velocity)
pd.DataFrame({name: np.linalg.norm(values, axis=1)
              for name, values in velocity.items() if np.asarray(values).shape == population.X.shape}).describe()

/data/cytobridge/projects/CytoBridge-ST-1104/envs/arista-api/lib/python3.10/site-packages/torch/autograd/graph.py:979: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:408.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


,drift,interaction,score,full
count,528.000000,528.000000,528.000000,528.000000
mean,5.242098,3.474513,0.016906,6.626749
std,1.216283,0.951647,0.003156,1.303803
min,2.483397,0.273564,0.007413,3.541809
25%,4.398686,2.798724,0.014782,5.683660
50%,5.077703,3.312709,0.016914,6.560394
75%,5.954815,4.026312,0.019109,7.473622
max,9.543683,8.898081,0.026578,11.125665


## Calculate growth

Evaluate growth on every observed population. The returned table includes the
time, cell type, coordinates and growth value. The same values are also stored
in each population's `obs['growth_rate']`.

In [4]:
slices = {str(float(t)): states[np.isclose(states.obs[time_key], t)].copy()
          for t in observed_times}
growth = cb.tl.evaluate_growth_by_timepoint(
    slices, model, time_points=[float(t) for t in observed_times],
    annotation_key=annotation_key, device=DEVICE,
)
growth.to_csv(OUTPUT_DIR / "growth_by_cell.csv", index=False)
growth.head()

,time,time_key,cell_index,x,y,growth,celltype
0,0.0,0.0,0,-0.309582,0.220557,0.787985,Valve cells
1,0.0,0.0,1,0.006353,0.207816,2.210886,Immature myocardial cells
2,0.0,0.0,2,-0.288070,0.175535,0.948449,Valve cells
3,0.0,0.0,3,-0.295561,-0.019695,1.824738,Immature myocardial cells
4,0.0,0.0,4,0.168691,0.021806,2.370483,Immature myocardial cells


Continue with [Simulate trajectories](../trajectory_analysis.md) using the
same `states` and `model`. The [chicken-heart notebook](dataset_workflows/chicken_heart.ipynb)
calculates and plots the growth summaries in Supplementary Fig. S9.
For velocity projections and cell-type transitions, continue to
[Chicken-heart trajectories and interactions](paper_figures/chicken_heart_daily.ipynb).
